# Ribbon Plot

A ribbon plot draws a shaded band around a central line. It is the standard way to present a point estimate together with its uncertainty — most often a forecast and its confidence or prediction interval. This tutorial uses the `ribbon` keyword in [Plots.jl](https://github.com/JuliaPlots/Plots.jl).

In [ ]:
using Plots
using Dates
using Random

Random.seed!(7);

## Symmetric Band

The simplest ribbon is symmetric: a single half-width is added above and below the central line. Here we wrap a fitted line in a constant ± one-standard-error band.

In [ ]:
n = 48
t = Date(2020, 1, 1) .+ Month.(0:n-1)

fitted = 5.0 .+ 0.2 .* (0:n-1)
se     = 1.5

plot(t, fitted;
    ribbon    = se,
    fillalpha = 0.2,
    lw        = 2,
    label     = "Fitted",
    xlabel    = "Date",
    ylabel    = "Value",
    title     = "Fitted Line with +/- 1 SE Band",
    legend     = :topleft)

The line is the point estimate; the shaded band communicates uncertainty. `fillalpha` controls transparency so the line beneath, and any overlapping series, remain readable.

## Forecast with a Widening Interval

The most common use of a ribbon is to show a forecast together with a prediction interval. Uncertainty almost always grows with the horizon, so the band widens as the forecast extends into the future.

We split the timeline into an observed history and a forecast horizon, then draw the forecast with a half-width that grows with the square root of the steps ahead — the characteristic shape of a random-walk prediction interval.

In [ ]:
h = 12
observed = 10 .+ cumsum(0.3 .+ randn(n))

last_value = observed[end]
fc      = last_value .+ 0.3 .* (1:h)
fc_time = t[end] .+ Month.(1:h)

# Prediction interval half-width grows with sqrt(horizon)
band = 1.96 .* sqrt.(1:h)

In [ ]:
plot(t, observed;
    lw     = 2,
    label  = "Observed",
    xlabel = "Date",
    ylabel = "Value",
    title  = "Forecast with 95% Prediction Interval",
    legend = :topleft)

plot!(fc_time, fc;
    ribbon    = band,
    fillalpha = 0.2,
    lw        = 2,
    linestyle = :dash,
    label     = "Forecast")

The band fans out with the horizon: the further ahead we forecast, the less certain we are. A band that stayed flat or shrank far into the future would be a warning sign of an overconfident model.

## Asymmetric Band

When the interval is not symmetric about the central line — for example a quantile-based interval — pass a tuple of `(lower, upper)` half-widths. The band then spans from `y - lower` to `y + upper`.

In [ ]:
lower = 1.0 .+ 0.5 .* sqrt.(1:h)
upper = 2.0 .+ 1.0 .* sqrt.(1:h)

plot(fc_time, fc;
    ribbon    = (lower, upper),
    fillalpha = 0.2,
    lw        = 2,
    label     = "Forecast",
    xlabel    = "Date",
    ylabel    = "Value",
    title     = "Asymmetric Interval",
    legend    = :topleft)

## Interpretation Cautions

A ribbon must have a defined meaning. Always state whether the band is:

- a **confidence interval** (uncertainty about a parameter or mean),
- a **prediction interval** (uncertainty about a future observation), or
- a fixed **+/- k standard deviations**.

A decorative band with no stated coverage misleads the reader into trusting precision that was never computed.

## Summary

- `ribbon = halfwidth` draws a symmetric band; `ribbon = (lower, upper)` draws an asymmetric one.
- Forecast bands should widen with the horizon.
- Use `fillalpha` to keep the central line and overlapping series legible.
- Only use a ribbon when the band has a precise, stated meaning.